
# DATA 304 — Module 7 Assignment: Tidying I (Reshaping & Missing Values)

**Topics:** `melt`, `pivot`, `pivot_table`, `stack`, `unstack`, MultiIndex tips, missing values detection and imputation.

**Rules**  
- Only edit the cells that begin with `# ASSIGNMENT CELL:`.  
- Do not change variable names requested by a question.  
- Run cells in order.  
- Use pandas, numpy only unless stated.


In [1]:
# ASSIGNMENT CELL: SETUP **DO NOT CHANGE ANYTHING IN THIS CELL**
import pandas as pd
import numpy as np

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', None)

# Synthetic datasets used across questions
# Wide sales by quarter, with some missing values
sales_wide = pd.DataFrame({
    'id':[101,102,103,104],
    'region':['N','S','N','W'],
    'q1':[10.0, np.nan, 9.0, 13.0],
    'q2':[11.0, 8.0, np.nan, 14.0],
    'q3':[np.nan, 7.0, 11.0, np.nan],
    'q4':[14.0, 9.0, 12.0, 16.0]
})

# Long treatment measurement
treat_long = pd.DataFrame({
    'name': ['John','John','Mary','Mary','Zoe','Zoe','Zoe'],
    'gender': ['M','M','F','F','F','F','F'],
    'treatment': ['A','B','A','B','A','A','B'],
    'value': [2,5,4,7,3,3,6]
})

# MultiIndex-like measurements for products
prod_long = pd.DataFrame({
    'id':[1,1,1,1,2,2,2,2],
    'product':['X','X','Y','Y','X','X','Y','Y'],
    'variable':['price','qty','price','qty','price','qty','price','qty'],
    'value':[5.0, 10, 7.5, 6, 5.5, 12, 8.0, 5]
})

### Q1. Melt wide quarterly sales into long format

Task: 
- Use pd.melt() to reshape sales_wide into a long format DataFrame named sales_long.
- Keep id and region as identifier columns (id_vars=['id','region']).
- Rename the variable column to quarter and the value column to sales.

In [2]:
# ASSIGNMENT CELL: Q1

# your code starts here
sales_long = sales_long = pd.melt(
    sales_wide,
    id_vars=['id','region'],
    var_name='quarter',
    value_name='sales'
)
# your code ends here

sales_long.head()

,id,region,quarter,sales
0,101,N,q1,10.0
1,102,S,q1,NaN
2,103,N,q1,9.0
3,104,W,q1,13.0
4,101,N,q2,11.0


### Q2. Pivot long back to wide

Task: 
- Pivot sales_long back to wide shape as sales_wide2
- Return a flat DataFrame with ordinary columns in natural quarter order q1..q4

In [3]:
# ASSIGNMENT CELL: Q2

# your code starts here
sales_wide2 = (sales_long
    .pivot(index=['id','region'], columns='quarter', values='sales')
    .reset_index()
)
# your code ends here

sales_wide2.head()

quarter,id,region,q1,q2,q3,q4
0,101,N,10.0,11.0,NaN,14.0
1,102,S,NaN,8.0,7.0,9.0
2,103,N,9.0,NaN,11.0,12.0
3,104,W,13.0,14.0,NaN,16.0


### Q3. Use pivot_table to handle duplicates by mean

Task: 
- From treat_long, compute average value per (name, gender) across treatments using pivot_table
- The resulting DataFrame avg_wide should have columns: ['name','gender','A','B']
- Tip: use aggfunc='mean'

In [4]:
# ASSIGNMENT CELL: Q3

# your code starts here
avg_wide = pd.pivot_table(
    treat_long,
    index=['name','gender'],
    columns='treatment',
    values='value',
    aggfunc='mean'
).reset_index()
# your code ends here

avg_wide.head()

treatment,name,gender,A,B
0,John,M,2.0,5.0
1,Mary,F,4.0,7.0
2,Zoe,F,3.0,6.0


### Q4. Stack to long Series then to tidy DataFrame

Task: 
- Begin with prod_long and first reshape it to a wide form, then convert it back to long.
    1. Create prod_wide where each (id, product) pair has columns price and qty.
    2. Use stack() to move the variable level into rows, producing a tidy DataFrame named tidy_prod.
- The final tidy_prod should have columns ['id', 'product', 'variable', 'value'].

In [5]:
# ASSIGNMENT CELL: Q4

# your code starts here
prod_wide = prod_long.pivot(index=['id','product'], columns='variable', values='value')
s = prod_wide.stack(future_stack=True)
s.name = 'value'
tidy_prod = s.reset_index()
# your code ends here

tidy_prod.head()

,id,product,variable,value
0,1,X,price,5.0
1,1,X,qty,10.0
2,1,Y,price,7.5
3,1,Y,qty,6.0
4,2,X,price,5.5


### Q5. Unstack long to wide by variable

Task: 
- Using prod_long, produce wide_by_var with columns ['id','product','price','qty']
- Use set_index([...])['value'].unstack('variable') and reset_index.

In [6]:
# ASSIGNMENT CELL: Q5

# your code starts here
wide_by_var = (prod_long
    .set_index(['id','product','variable'])['value']
    .unstack('variable')
    .reset_index()
)
# your code ends here

wide_by_var.head()

variable,id,product,price,qty
0,1,X,5.0,10.0
1,1,Y,7.5,6.0
2,2,X,5.5,12.0
3,2,Y,8.0,5.0


### Q6. Detect missingness by column

Task: 
- Compute a Series miss_counts with count of NaN per column in sales_wide

In [7]:
# ASSIGNMENT CELL: Q6

# your code starts here
miss_counts = sales_wide.isna().sum()
# your code ends here

miss_counts

id        0
region    0
q1        1
q2        1
q3        2
q4        0
dtype: int64

### Q7. Drop rows missing q1 but keep others

Task: 
- Create sales_drop_q1 from sales_wide by dropping rows where q1 is NaN, but keep other NaNs.

In [8]:
# ASSIGNMENT CELL: Q7

# your code starts here
sales_drop_q1 = sales_wide.dropna(subset=['q1'])
# your code ends here

sales_drop_q1

,id,region,q1,q2,q3,q4
0,101,N,10.0,11.0,NaN,14.0
2,103,N,9.0,NaN,11.0,12.0
3,104,W,13.0,14.0,NaN,16.0


### Q8. Forward-fill within each id across q1→q4 order

Task: 
- Map quarters to order {'q1':1,'q2':2,'q3':3,'q4':4}.
- Within each id, sort by this order.
- Forward-fill then back-fill sales only within the same id.
- Do not change originally non-missing values.
- Return ffilled with columns ['id','region','quarter','sales'].

In [9]:
# ASSIGNMENT CELL: Q8

order = {'q1':1,'q2':2,'q3':3,'q4':4}

# your code starts here
ffilled = (
    sales_long
      .assign(q_order=lambda d: d['quarter'].map(order))
      .sort_values(['id','q_order'])
      .assign(sales=lambda d: d.groupby('id')['sales'].ffill().bfill())
      .drop(columns='q_order')
      .reset_index(drop=True)
)
# your code ends here

ffilled.head()

,id,region,quarter,sales
0,101,N,q1,10.0
1,101,N,q2,11.0
2,101,N,q3,11.0
3,101,N,q4,14.0
4,102,S,q1,8.0


### Q9. Linear interpolation across quarters per id

Task: 
- Create sales_interp from sales_long by linearly interpolating the sales values within each id, ordered by quarter (q1 → q4).
- Use the quarter order mapping {'q1':1,'q2':2,'q3':3,'q4':4} to control sequence.
- Preserve existing NaN values at the beginning or end of each group when interpolation is not possible.
- The output sales_interp must have the same columns and row count as sales_long.

In [10]:
# ASSIGNMENT CELL: Q9

# your code starts here
sales_interp = (
    sales_long
      .assign(q_order=lambda d: d['quarter'].map(order))
      .sort_values(['id','q_order'])
      .assign(sales=lambda d: d.groupby('id')['sales'].transform(lambda s: s.interpolate()))
      .drop(columns='q_order')
      .reset_index(drop=True)
)
# your code ends here

sales_interp.head()

,id,region,quarter,sales
0,101,N,q1,10.0
1,101,N,q2,11.0
2,101,N,q3,12.5
3,101,N,q4,14.0
4,102,S,q1,NaN


### Q10. Robust pivot with duplicates using pivot_table

Task: 
- From treat_long, build treat_wide_mean with columns ['name','gender','A','B'] using pivot_table mean.

In [11]:
# ASSIGNMENT CELL: Q10

# your code starts here
treat_wide_mean = (pd.pivot_table(
    treat_long,
    index=['name','gender'],
    columns='treatment',
    values='value',
    aggfunc='mean'
).reset_index())
# your code ends here

treat_wide_mean.head()

treatment,name,gender,A,B
0,John,M,2.0,5.0
1,Mary,F,4.0,7.0
2,Zoe,F,3.0,6.0
